# CNN ensemble trained on the pop-cosmos mock

The usual 1-parameter CNN ensemble (`znn.train_ensembles`, K-fold, MSE), trained and tested on the
pop-cosmos mock in its native COSMOS2020 bands, encoded on the LSST+Roman wavelength grid (see
`popcosmos_encoding.ipynb` for the encoding itself).

- **Bands:** the 23 COSMOS2020 bands inside the LSST ugrizy + Roman Y106/J129/H158 range (Ks and IRAC
  dropped); `lambda_range` fixed to that range, so the inputs share their bins with the challenge data.
- **Data:** disjoint random train and test samples of the mock with HSC i < `I_MAX`. The photometry is
  the mock's own noisy fluxes, at COSMOS2020 depth (much deeper than LSST in the optical).
- **Output:** the ensemble is saved with `znn.save_ensemble_file` (members, bands, config with
  `lambda_range`, filter curves) to `MODEL_FILE`, and reloaded on re-runs. It is the pre-trained model for
  `CNN_photoz-ensemble-cardinal_popcosmos_pretraining.ipynb`.

In [1]:
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qp.metrics.pit import PIT

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

import znn  # run from the repo root so this picks up the local source

I0000 00:00:1791378270.057440  535339 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791378270.058934  535339 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791378270.163410  535339 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791378273.173360  535339 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

## Configuration

In [2]:
MOCK_FILE = "mock_catalog_Ch1_26.h5"
FILTER_DIR = "data/popcosmos_filters"
MODEL_FILE = os.environ.get("CNNPZ_POPCOSMOS_MODEL", "models/popcosmos/popcosmos_ensemble_CNN_6layers.pkl")

I_MAX = 25.5  # magnitude cut in HSC i
N_TRAIN = 200_000  # training objects (split K ways inside train_ensembles)
N_TEST = 50_000  # held-out test objects, disjoint from the training sample
SEED = 42

N_SPLITS = 5  # ensemble members (K-fold); fine-tuning uses one fold per member
EPOCHS = 100  # max epochs per member (early stopping usually ends sooner)

# (band, filter file) in the mock's column order; see data/popcosmos_filters/SOURCES.md
MOCK_BANDS = [
    ("u", "u_megaprime_sagem.res"),
    ("g", "hsc_g.res"),
    ("r", "hsc_r.res"),
    ("i", "hsc_i.res"),
    ("z", "hsc_z.res"),
    ("y", "hsc_y.par"),
    ("Y", "uvista_y_cosmos.par"),
    ("J", "uvista_j_cosmos.par"),
    ("H", "uvista_h_cosmos.par"),
    ("Ks", "uvista_ks_cosmos.par"),
    *[
        (name, f"ia{name[2:]}_cosmos.par")
        for name in ["IB427", "IB464", "IA484", "IB505", "IA527", "IB574",
                     "IA624", "IA679", "IB709", "IA738", "IA767", "IB827"]
    ],
    ("NB711", "NB711.SuprimeCam.par"),
    ("NB816", "NB816.SuprimeCam.par"),
    ("ch1", "irac1_cosmos.par"),
    ("ch2", "irac2_cosmos.par"),
]
MOCK_COLUMN = {band: k for k, (band, _) in enumerate(MOCK_BANDS)}

# Wavelength span of the DC2LSST_{ugrizy} and roman_{Y106,J129,H158} curves used for the challenge
LAMBDA_RANGE = (3000.0, 18650.0)  # Å
REF_BAND = "i"
CONFIG = {**znn.DEFAULT_FEATURE_CONFIG, "lambda_range": LAMBDA_RANGE}


def response_range(curve, level=0.01):
    w, t = curve.T
    w = w[t > level * t.max()]
    return w.min(), w.max()


# keep the bands whose response (> 1% of peak) lies inside LAMBDA_RANGE
BANDS = {}
for band, file in MOCK_BANDS:
    lo, hi = response_range(np.loadtxt(f"{FILTER_DIR}/{file}"))
    if LAMBDA_RANGE[0] <= lo and hi <= LAMBDA_RANGE[1]:
        BANDS[band] = (f"mag_{band}", f"{FILTER_DIR}/{file}")
print(f"{len(BANDS)} bands:", list(BANDS))

23 bands: ['u', 'g', 'r', 'i', 'z', 'y', 'Y', 'J', 'H', 'IB427', 'IB464', 'IA484', 'IB505', 'IA527', 'IB574', 'IA624', 'IA679', 'IB709', 'IA738', 'IA767', 'IB827', 'NB711', 'NB816']


## Load and encode the mock

Fluxes are in maggies; negative noisy fluxes become non-detections (`NaN`, treated as unobserved).
Redshift is column 15 of `sps_parameters`.

In [3]:
with h5py.File(MOCK_FILE, "r") as f:
    fluxes = f["fluxes"][:]
    sigmas = f["flux_sigmas"][:]
    redshift = f["sps_parameters"][:, 15]

with np.errstate(divide="ignore", invalid="ignore"):
    detected = fluxes > 0
    mags = np.where(detected, -2.5 * np.log10(fluxes), np.nan)
    mag_errs = np.where(detected, 2.5 / np.log(10) * sigmas / fluxes, np.nan)

passing = np.flatnonzero(mags[:, MOCK_COLUMN[REF_BAND]] < I_MAX)
chosen = np.random.default_rng(SEED).permutation(passing)
idx = {"train": np.sort(chosen[:N_TRAIN]), "test": np.sort(chosen[N_TRAIN:N_TRAIN + N_TEST])}


def mock_catalog(rows):
    return pd.DataFrame(
        {
            **{column: mags[rows, MOCK_COLUMN[b]] for b, (column, _) in BANDS.items()},
            **{f"{column}_err": mag_errs[rows, MOCK_COLUMN[b]] for b, (column, _) in BANDS.items()},
            "redshift": redshift[rows],
        }
    )


train = mock_catalog(idx["train"])
test = mock_catalog(idx["test"])
print(f"{len(passing):,} mock galaxies with i < {I_MAX}; train {len(train):,}, test {len(test):,}")
del fluxes, sigmas, mags, mag_errs

filter_curves = znn.load_filter_curves(BANDS)
X_train, Y_train = znn.catalog_to_XY(train, BANDS, REF_BAND, CONFIG, filter_curves=filter_curves)
X_test, Y_test = znn.catalog_to_XY(test, BANDS, REF_BAND, CONFIG, filter_curves=filter_curves)
mag_i_test = test[f"mag_{REF_BAND}"].to_numpy()
print("train:", X_train.shape, "test:", X_test.shape)

1,048,241 mock galaxies with i < 25.5; train 200,000, test 50,000
train: (200000, 32, 3) test: (50000, 32, 3)


## Train the ensemble (or load it if already trained)

In [4]:
if os.path.exists(MODEL_FILE):
    trained_models, features = znn.load_ensemble_file(MODEL_FILE)
    assert features["config"] == CONFIG and list(features["bands"]) == list(BANDS), "MODEL_FILE was trained with other settings"
    histories = None
else:
    trained_models, histories = znn.train_ensembles(znn.build_model, X_train, Y_train, N_SPLITS=N_SPLITS, EPOCHS=EPOCHS)
    os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)
    znn.save_ensemble_file(MODEL_FILE, trained_models, BANDS, REF_BAND, CONFIG)

if histories is not None:
    znn.plot_ensemble_losses(histories, ylim=(0, 0.002))


--- Fold 1/5 ---


/home/jaimerzp/anaconda3/envs/qp/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1791378279.567129  535339 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/100
625/625 - 60s - 96ms/step - loss: 0.0210 - val_loss: 0.0059 - learning_rate: 0.0010
Epoch 2/100
625/625 - 79s - 127ms/step - loss: 0.0076 - val_loss: 0.0038 - learning_rate: 0.0010
Epoch 3/100
625/625 - 53s - 85ms/step - loss: 0.0061 - val_loss: 0.0039 - learning_rate: 0.0010
Epoch 4/100
625/625 - 53s - 85ms/step - loss: 0.0053 - val_loss: 0.0029 - learning_rate: 0.0010
Epoch 5/100
625/625 - 51s - 81ms/step - loss: 0.0047 - val_loss: 0.0025 - learning_rate: 0.0010
Epoch 6/100
625/625 - 53s - 84ms/step - loss: 0.0044 - val_loss: 0.0024 - learning_rate: 0.0010
Epoch 7/100
625/625 - 51s - 82ms/step - loss: 0.0043 - val_loss: 0.0026 - learning_rate: 0.0010
Epoch 8/100
625/625 - 29s - 47ms/step - loss: 0.0044 - val_loss: 0.0028 - learning_rate: 0.0010
Epoch 9/100
625/625 - 54s - 87ms/step - loss: 0.0042 - val_loss: 0.0022 - learning_rate: 0.0010
Epoch 10/100
625/625 - 54s - 86ms/step - loss: 0.0039 - val_loss: 0.0031 - learning_rate: 0.0010
Epoch 11/100
625/625 - 56s - 89ms/step

KeyboardInterrupt: 

## Test-set performance

p(z) from the ensemble spread (`znn.ensemble_predict`), the plain version: the mock's errors are
heavier-tailed than its quoted `flux_sigmas` (see `popcosmos_encoding.ipynb`), so resampled prediction
is left for later. Δz = (z_phot − z_true) / (1 + z_true), with the p(z) mean as z_phot.

In [ ]:
pz = znn.ensemble_predict(trained_models, X_test)
z_point = np.squeeze(pz.ancil["zmode"])
width = znn.predictions_mean_std(pz)[1].ravel()

redshift_bins = np.linspace(0, 4, 11)
imag_bins = np.linspace(18, I_MAX, 11)
stats = znn.get_all_stats(Y_test, z_point, mag_i_test, save=False, redshift_bins=redshift_bins, imag_bins=imag_bins)
znn.plot_stats(*stats, Y_test, z_point, redshift_bins, imag_bins, mag_i_test)
plt.show()

bias, bias_err, scatter, outlier_rate, abs_outlier_rate = stats[0]
print(f"bias {bias:+.4f}, scatter {scatter:.4f}, outlier rate (3 sigma) {outlier_rate:.4f}, |dz| > 0.2 rate {abs_outlier_rate:.4f}")

## p(z) calibration: PIT

As in the Cardinal notebooks: the ensemble spread only measures model disagreement, so expect p(z)
that is too narrow (a U-shaped PIT).

In [ ]:
pit = PIT(pz, Y_test)
pit_stats = {
    "ks": pit.evaluate_PIT_KS().statistic,
    "CvM": pit.evaluate_PIT_CvM().statistic,
    "ksamp": pit.evaluate_PIT_anderson_ksamp().statistic,
    "outlier": np.mean((pit.pit_samps < 0.0001) | (pit.pit_samps > 0.9999)),
}

fig, axarr = plt.subplots(1, 2, figsize=[13, 5])
axarr[0].hist(pit.pit_samps, bins=50, range=(0, 1), density=True, histtype="step", lw=2)
axarr[0].axhline(1, color="k", ls="--")
axarr[0].set_xlabel("PIT")
axarr[0].set_ylabel("density")
q = np.linspace(0, 1, 101)
axarr[1].plot(q, np.quantile(pit.pit_samps, q), lw=2)
axarr[1].plot([0, 1], [0, 1], "k--")
axarr[1].set_xlabel("uniform quantile")
axarr[1].set_ylabel("PIT quantile")
plt.tight_layout()
plt.show()

pd.Series({**pit_stats, "median p(z) width": np.median(width)}).round(4)